# Bangla Human vs AI Text Classification with Clean Development–Holdout Evaluation

This notebook follows a clean workflow for model comparison and ensemble learning:

1. Data cleaning and final merged dataframe preparation  
2. Groupwise 80/20 development–holdout split  
3. Feature extraction for development and holdout sets  
4. Baseline six-model grouped cross-validation comparison  
5. Hyperparameter tuning of the top three models and final top-two selection  
6. OOF stacking and final holdout test evaluation  

**Important evaluation rule:** the holdout test set stays untouched until the final section.


In [1]:
import importlib
import subprocess
import sys

def ensure_package(import_name, pip_name=None):
    pip_name = pip_name or import_name
    try:
        importlib.import_module(import_name)
        print(f"{pip_name} is already available")
    except ImportError:
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

REQUIRED_PACKAGES = [
    ("transformers", "transformers"),
    ("sentencepiece", "sentencepiece"),
    ("xgboost", "xgboost"),
    ("lightgbm", "lightgbm"),
]

for import_name, pip_name in REQUIRED_PACKAGES:
    ensure_package(import_name, pip_name)


transformers is already available
sentencepiece is already available
xgboost is already available
lightgbm is already available


In [2]:
import os
import re
import math
import json
import warnings
import unicodedata
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
from transformers import AutoTokenizer, AutoModel

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    GroupShuffleSplit,
    GroupKFold,
    GridSearchCV,
    cross_validate,
)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    roc_curve,
    auc,
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)


## Section 1: Data Cleaning and Final Merged Dataframe Preparation

### 1.1 Setup and paths


In [3]:
IS_KAGGLE = Path("/kaggle/input").exists()
print("Running on Kaggle:", IS_KAGGLE)

PREFERRED_DATASET_FOLDER = None   # Example: "my-dataset-folder"
DATA_FILENAME = "merge dataset V2.csv"

OUTPUT_ROOT = Path("/kaggle/working/outputs" if Path("/kaggle/working").exists() else "./outputs")
PLOT_DIR = OUTPUT_ROOT / "plots"
TABLE_DIR = OUTPUT_ROOT / "tables"
PRED_DIR = OUTPUT_ROOT / "predictions"
FEATURE_DIR = OUTPUT_ROOT / "features"

for folder in [OUTPUT_ROOT, PLOT_DIR, TABLE_DIR, PRED_DIR, FEATURE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

HUMAN_TEXT_COL = "Context(Human)"
AI_TEXT_COL = "Context(ChatGpt 1st paraphrase)"

RANDOM_STATE = 42
HOLDOUT_TEST_SIZE = 0.20
N_SPLITS_CV = 5
BATCH_SIZE = 16
MAX_LENGTH = 512
MODEL_NAME = "csebuetnlp/banglabert"

print("OUTPUT_ROOT:", OUTPUT_ROOT)

def find_dataset_file(filename=DATA_FILENAME, preferred_folder=PREFERRED_DATASET_FOLDER):
    search_roots = [
        Path("/kaggle/input"),
        Path("/kaggle/working"),
        Path("."),
        Path("/mnt/data"),
    ]

    if preferred_folder:
        preferred_candidates = [
            Path("/kaggle/input") / preferred_folder / filename,
            Path(preferred_folder) / filename,
        ]
        for candidate in preferred_candidates:
            if candidate.exists():
                return candidate

    for root in search_roots:
        if root.exists():
            matches = list(root.rglob(filename))
            if matches:
                return matches[0]

    raise FileNotFoundError(
        f"Could not find '{filename}'. Attach the dataset in Kaggle input or set PREFERRED_DATASET_FOLDER."
    )

DATA_PATH = find_dataset_file()
print("Dataset found at:", DATA_PATH)


Running on Kaggle: True
OUTPUT_ROOT: /kaggle/working/outputs


FileNotFoundError: Could not find 'merge dataset V2.csv'. Attach the dataset in Kaggle input or set PREFERRED_DATASET_FOLDER.

### 1.2 Load raw data and basic cleaning


In [ ]:
df_raw = pd.read_csv(DATA_PATH)
print("Raw shape:", df_raw.shape)

required_cols = [HUMAN_TEXT_COL, AI_TEXT_COL]
missing_cols = [col for col in required_cols if col not in df_raw.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df = df_raw.copy()

print("Duplicate full rows before drop:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print("Duplicate full rows after drop :", df.duplicated().sum())

df = df.dropna(subset=required_cols).reset_index(drop=True)
df[HUMAN_TEXT_COL] = df[HUMAN_TEXT_COL].astype(str)
df[AI_TEXT_COL] = df[AI_TEXT_COL].astype(str)

df["source_id"] = np.arange(len(df))
print("Shape after basic cleaning:", df.shape)

df.head()


### 1.3 Text normalization


In [ ]:
try:
    from normalizer import normalize as buet_normalize
    NORMALIZER_STATUS = "already available"
except Exception:
    try:
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "git+https://github.com/csebuetnlp/normalizer",
        ])
        from normalizer import normalize as buet_normalize
        NORMALIZER_STATUS = "installed from GitHub"
    except Exception:
        buet_normalize = None
        NORMALIZER_STATUS = "fallback normalization only"

print("Normalizer status:", NORMALIZER_STATUS)

def normalize_text(text):
    if not isinstance(text, str):
        return ""
    text = text.strip()
    if buet_normalize is not None:
        try:
            text = buet_normalize(text)
        except Exception:
            pass
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"[\u200B-\u200D\uFEFF]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["normalized_human_text"] = df[HUMAN_TEXT_COL].apply(normalize_text)
df["normalized_ai_text"] = df[AI_TEXT_COL].apply(normalize_text)

print("Duplicate normalized human text:", df.duplicated(subset=["normalized_human_text"]).sum())
print("Duplicate normalized AI text   :", df.duplicated(subset=["normalized_ai_text"]).sum())

df = df.drop_duplicates(subset=["normalized_human_text"]).reset_index(drop=True)
df = df.drop_duplicates(subset=["normalized_ai_text"]).reset_index(drop=True)

same_text_count = (df["normalized_human_text"] == df["normalized_ai_text"]).sum()
print("Exact human==AI pairs before remove:", same_text_count)

df = df[df["normalized_human_text"] != df["normalized_ai_text"]].reset_index(drop=True)

print("Final pair-level shape after normalization and filtering:", df.shape)


### 1.4 Final merged dataframe


In [ ]:
human_df = pd.DataFrame({
    "source_id": df["source_id"],
    "normalized_text": df["normalized_human_text"],
    "label": "Human",
})

ai_df = pd.DataFrame({
    "source_id": df["source_id"],
    "normalized_text": df["normalized_ai_text"],
    "label": "AI",
})

merged_df = pd.concat([human_df, ai_df], ignore_index=True)
merged_df = merged_df.dropna(subset=["normalized_text"]).reset_index(drop=True)
merged_df = merged_df[merged_df["normalized_text"].str.len() > 0].reset_index(drop=True)
merged_df["label_encoded"] = merged_df["label"].map({"Human": 0, "AI": 1}).astype(int)

print("Merged shape:", merged_df.shape)
print(merged_df["label"].value_counts())
print("Duplicate rows in merged_df:", merged_df.duplicated().sum())
print("Duplicate normalized_text in merged_df:", merged_df.duplicated(subset=["normalized_text"]).sum())

merged_df.to_csv(TABLE_DIR / "merged_df_final.csv", index=False)
merged_df.head()


## Section 2: Groupwise 80/20 Development–Holdout Split

### 2.1 Split the merged dataframe


In [ ]:
X_all = merged_df["normalized_text"]
y_all = merged_df["label_encoded"]
groups_all = merged_df["source_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=HOLDOUT_TEST_SIZE,
    random_state=RANDOM_STATE,
)

dev_idx, holdout_idx = next(gss.split(X_all, y_all, groups=groups_all))

dev_df = merged_df.iloc[dev_idx].reset_index(drop=True)
holdout_test_df = merged_df.iloc[holdout_idx].reset_index(drop=True)

print("Development shape :", dev_df.shape)
print("Holdout test shape:", holdout_test_df.shape)
print()
print("Development label distribution:")
print(dev_df["label"].value_counts())
print()
print("Holdout test label distribution:")
print(holdout_test_df["label"].value_counts())
print()
print("Common source_id between dev and holdout:",
      len(set(dev_df["source_id"]).intersection(set(holdout_test_df["source_id"]))))

dev_df.to_csv(TABLE_DIR / "dev_df_groupwise.csv", index=False)
holdout_test_df.to_csv(TABLE_DIR / "holdout_test_df_groupwise.csv", index=False)


## Section 3: Feature Extraction for Development and Holdout Sets

### 3.1 BanglaBERT embedding setup


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
embedding_model = AutoModel.from_pretrained(MODEL_NAME).to(device)
embedding_model.eval()

def get_embeddings_mean_pool(texts, batch_size=BATCH_SIZE, max_length=MAX_LENGTH):
    all_embeddings = []

    for start in tqdm(range(0, len(texts), batch_size), desc="Extracting BanglaBERT embeddings"):
        batch_texts = texts[start:start + batch_size]

        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        ).to(device)

        with torch.no_grad():
            outputs = embedding_model(**inputs)
            last_hidden = outputs.last_hidden_state
            attention_mask = inputs["attention_mask"].unsqueeze(-1).float()

            masked_hidden = last_hidden * attention_mask
            summed = masked_hidden.sum(dim=1)
            counts = attention_mask.sum(dim=1).clamp(min=1.0)
            mean_pool = summed / counts

            all_embeddings.append(mean_pool.cpu().numpy().astype(np.float32))

    return np.vstack(all_embeddings)


### 3.2 Stylometric feature extraction setup


In [ ]:
BANGLA_STOPWORDS = {
    "এই", "ওই", "সে", "তিনি", "তারা", "আমরা", "আমি", "তুমি", "আপনি",
    "এবং", "আর", "কিন্তু", "তবে", "যদি", "যেন", "কারণ", "তাই",
    "একটি", "একটা", "কিছু", "অনেক", "সব", "সবই",
    "হয়", "হবে", "ছিল", "ছিলো", "আছে", "থাকে", "থাকবে", "করেছে", "করে",
    "না", "নয়", "তো", "ও", "ই", "কি", "কেন", "কী",
    "জন্য", "দিকে", "পর", "পরে", "সাথে", "সঙ্গে", "মধ্যে", "উপর", "নিচে",
    "থেকে", "দিয়ে", "এ", "তে", "রা", "র", "এর", "কে"
}

BANGLA_VOWELS = set("অআইঈউঊঋএঐওঔািীুূৃেৈোৌ")

def safe_div(a, b):
    return a / b if b != 0 else 0.0

def shannon_entropy(text):
    if not text:
        return 0.0
    counts = Counter(text)
    total = len(text)
    entropy = 0.0
    for count in counts.values():
        p = count / total
        entropy -= p * math.log2(p)
    return entropy

def normalize_spaces(text):
    text = "" if text is None else str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def split_sentences(text):
    sentences = re.split(r"[।!?]+", text)
    return [s.strip() for s in sentences if s.strip()]

def tokenize_words(text):
    return re.findall(r"[\u0980-\u09FFA-Za-z0-9]+", text)

def get_ngrams(tokens, n):
    return list(zip(*[tokens[i:] for i in range(n)])) if len(tokens) >= n else []

def count_regex(pattern, text):
    return len(re.findall(pattern, text))

def script_type(word):
    has_bangla = bool(re.search(r"[\u0980-\u09FF]", word))
    has_latin = bool(re.search(r"[A-Za-z]", word))
    has_digit = bool(re.search(r"\d", word))

    if has_bangla and not has_latin and not has_digit:
        return "bangla"
    if has_latin and not has_bangla and not has_digit:
        return "latin"
    if has_digit and not has_bangla and not has_latin:
        return "digit"
    return "mixed"

def describe_lengths(values):
    if not values:
        return {"min": 0.0, "max": 0.0, "mean": 0.0, "std": 0.0, "median": 0.0}
    arr = np.array(values, dtype=np.float32)
    return {
        "min": float(arr.min()),
        "max": float(arr.max()),
        "mean": float(arr.mean()),
        "std": float(arr.std()),
        "median": float(np.median(arr)),
    }

def extract_stylometric_features(text, stopwords=BANGLA_STOPWORDS):
    text = normalize_spaces(text)

    words = tokenize_words(text)
    sentences = split_sentences(text)

    char_count = len(text)
    non_space_char_count = len(text.replace(" ", ""))
    word_count = len(words)
    sent_count = len(sentences)

    word_lengths = [len(w) for w in words]
    sent_lengths = [len(tokenize_words(s)) for s in sentences]

    word_len_stats = describe_lengths(word_lengths)
    sent_len_stats = describe_lengths(sent_lengths)

    word_counter = Counter(words)
    unique_words = len(word_counter)
    hapax_count = sum(1 for c in word_counter.values() if c == 1)
    dislegomena_count = sum(1 for c in word_counter.values() if c == 2)
    top_word_freq = max(word_counter.values()) if word_counter else 0
    avg_token_frequency = float(np.mean(list(word_counter.values()))) if word_counter else 0.0

    bigrams = get_ngrams(words, 2)
    trigrams = get_ngrams(words, 3)
    unique_bigrams = len(set(bigrams))
    unique_trigrams = len(set(trigrams))

    char_counter = Counter(text)
    unique_chars = len(char_counter)

    bangla_chars = re.findall(r"[\u0980-\u09FF]", text)
    latin_chars = re.findall(r"[A-Za-z]", text)
    digit_chars = re.findall(r"\d", text)
    whitespace_chars = re.findall(r"\s", text)
    punct_chars = re.findall(r"[,\.;:!?।\"'“”‘’()\[\]{}\-—/…]", text)
    vowel_chars = [c for c in bangla_chars if c in BANGLA_VOWELS]

    comma_count = count_regex(r",", text)
    danda_count = count_regex(r"।", text)
    question_count = count_regex(r"\?", text)
    exclam_count = count_regex(r"!", text)
    colon_count = count_regex(r":", text)
    semicolon_count = count_regex(r";", text)
    quote_count = count_regex(r"[\"'“”‘’]", text)
    bracket_count = count_regex(r"[\(\)\[\]\{\}]", text)
    hyphen_count = count_regex(r"[\-—]", text)
    slash_count = count_regex(r"/", text)
    ellipsis_count = count_regex(r"…|\.\.\.", text)
    repeated_punct_count = count_regex(r"([!?.,])\1+", text)

    stopword_count = sum(1 for w in words if w in stopwords)
    repeated_word_count = sum(c for c in word_counter.values() if c > 1)
    short_word_count = sum(1 for w in words if len(w) <= 3)
    medium_word_count = sum(1 for w in words if 4 <= len(w) <= 6)
    long_word_count = sum(1 for w in words if len(w) >= 7)

    script_counts = Counter(script_type(w) for w in words)

    return {
        "char_count": char_count,
        "non_space_char_count": non_space_char_count,
        "word_count": word_count,
        "sentence_count": sent_count,

        "min_word_length": word_len_stats["min"],
        "max_word_length": word_len_stats["max"],
        "mean_word_length": word_len_stats["mean"],
        "std_word_length": word_len_stats["std"],
        "median_word_length": word_len_stats["median"],

        "min_sentence_length": sent_len_stats["min"],
        "max_sentence_length": sent_len_stats["max"],
        "mean_sentence_length": sent_len_stats["mean"],
        "std_sentence_length": sent_len_stats["std"],
        "median_sentence_length": sent_len_stats["median"],

        "chars_per_word": safe_div(non_space_char_count, word_count),
        "chars_per_sentence": safe_div(non_space_char_count, sent_count),
        "words_per_sentence": safe_div(word_count, sent_count),

        "type_token_ratio": safe_div(unique_words, word_count),
        "hapax_ratio": safe_div(hapax_count, word_count),
        "dislegomena_ratio": safe_div(dislegomena_count, word_count),
        "repeated_word_ratio": safe_div(repeated_word_count, word_count),
        "top_word_ratio": safe_div(top_word_freq, word_count),
        "avg_token_frequency": avg_token_frequency,

        "unique_bigram_ratio": safe_div(unique_bigrams, len(bigrams)),
        "unique_trigram_ratio": safe_div(unique_trigrams, len(trigrams)),

        "unique_char_ratio": safe_div(unique_chars, char_count),
        "char_entropy": shannon_entropy(text),

        "bangla_char_ratio": safe_div(len(bangla_chars), char_count),
        "latin_char_ratio": safe_div(len(latin_chars), char_count),
        "digit_char_ratio": safe_div(len(digit_chars), char_count),
        "whitespace_ratio": safe_div(len(whitespace_chars), char_count),
        "punctuation_ratio": safe_div(len(punct_chars), char_count),
        "bangla_vowel_ratio": safe_div(len(vowel_chars), max(1, len(bangla_chars))),

        "comma_count": comma_count,
        "danda_count": danda_count,
        "question_count": question_count,
        "exclam_count": exclam_count,
        "colon_count": colon_count,
        "semicolon_count": semicolon_count,
        "quote_count": quote_count,
        "bracket_count": bracket_count,
        "hyphen_count": hyphen_count,
        "slash_count": slash_count,
        "ellipsis_count": ellipsis_count,
        "repeated_punct_count": repeated_punct_count,

        "punctuation_per_word": safe_div(len(punct_chars), word_count),
        "punctuation_per_sentence": safe_div(len(punct_chars), sent_count),

        "stopword_ratio": safe_div(stopword_count, word_count),
        "short_word_ratio": safe_div(short_word_count, word_count),
        "medium_word_ratio": safe_div(medium_word_count, word_count),
        "long_word_ratio": safe_div(long_word_count, word_count),

        "bangla_word_ratio": safe_div(script_counts["bangla"], word_count),
        "latin_word_ratio": safe_div(script_counts["latin"], word_count),
        "digit_word_ratio": safe_div(script_counts["digit"], word_count),
        "mixed_word_ratio": safe_div(script_counts["mixed"], word_count),
    }

def build_stylometry_dataframe(texts):
    rows = [extract_stylometric_features(text) for text in tqdm(texts, desc="Extracting stylometric features")]
    return pd.DataFrame(rows)

print("Stylometric feature setup ready.")


### 3.3 Development set feature extraction


In [ ]:
X_dev_text = dev_df["normalized_text"].tolist()
X_holdout_text = holdout_test_df["normalized_text"].tolist()

y_dev = dev_df["label_encoded"].astype(int).to_numpy()
y_holdout = holdout_test_df["label_encoded"].astype(int).to_numpy()
groups_dev = dev_df["source_id"].to_numpy()

X_dev_emb = get_embeddings_mean_pool(X_dev_text)
X_dev_style_df = build_stylometry_dataframe(X_dev_text)
stylometric_feature_names = X_dev_style_df.columns.tolist()
X_dev_style = X_dev_style_df.to_numpy(dtype=np.float32)
X_dev_fusion = np.concatenate([X_dev_emb, X_dev_style], axis=1)

print("Development embedding shape :", X_dev_emb.shape)
print("Development stylometry shape:", X_dev_style.shape)
print("Development fusion shape    :", X_dev_fusion.shape)


### 3.4 Holdout test feature extraction


In [ ]:
X_holdout_emb = get_embeddings_mean_pool(X_holdout_text)
X_holdout_style_df = build_stylometry_dataframe(X_holdout_text)
X_holdout_style_df = X_holdout_style_df[stylometric_feature_names]
X_holdout_style = X_holdout_style_df.to_numpy(dtype=np.float32)
X_holdout_fusion = np.concatenate([X_holdout_emb, X_holdout_style], axis=1)

print("Holdout embedding shape :", X_holdout_emb.shape)
print("Holdout stylometry shape:", X_holdout_style.shape)
print("Holdout fusion shape    :", X_holdout_fusion.shape)


### 3.5 Package and save feature sets


In [ ]:
FEATURE_SETS = {
    "BanglaBERT_Embedding_Only": {
        "dev": X_dev_emb,
        "holdout": X_holdout_emb,
    },
    "Stylometric_Features_Only": {
        "dev": X_dev_style,
        "holdout": X_holdout_style,
    },
    "Fusion_BanglaBERT_Plus_Stylometry": {
        "dev": X_dev_fusion,
        "holdout": X_holdout_fusion,
    },
}

shape_rows = []
for feature_name, feature_pack in FEATURE_SETS.items():
    shape_rows.append({
        "Feature_Set": feature_name,
        "Dev_Shape": tuple(feature_pack["dev"].shape),
        "Holdout_Shape": tuple(feature_pack["holdout"].shape),
    })

feature_shape_df = pd.DataFrame(shape_rows)
feature_shape_df.to_csv(TABLE_DIR / "feature_set_shapes.csv", index=False)

np.save(FEATURE_DIR / "X_dev_emb.npy", X_dev_emb)
np.save(FEATURE_DIR / "X_dev_style.npy", X_dev_style)
np.save(FEATURE_DIR / "X_dev_fusion.npy", X_dev_fusion)
np.save(FEATURE_DIR / "X_holdout_emb.npy", X_holdout_emb)
np.save(FEATURE_DIR / "X_holdout_style.npy", X_holdout_style)
np.save(FEATURE_DIR / "X_holdout_fusion.npy", X_holdout_fusion)

with open(TABLE_DIR / "stylometric_feature_names.txt", "w", encoding="utf-8") as f:
    for name in stylometric_feature_names:
        f.write(name + "\n")

feature_shape_df


## Section 4: Baseline Six-Model Grouped Cross-Validation Comparison

### 4.1 Baseline model configuration


In [ ]:
def make_baseline_model_specs(random_state=RANDOM_STATE):
    return {
        "Logistic Regression": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=3000, random_state=random_state)),
        ]),
        "Linear SVM": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LinearSVC(random_state=random_state)),
        ]),
        "Random Forest": RandomForestClassifier(
            n_estimators=400,
            random_state=random_state,
            n_jobs=-1,
        ),
        "Extra Trees": ExtraTreesClassifier(
            n_estimators=400,
            random_state=random_state,
            n_jobs=-1,
        ),
        "XGBoost": XGBClassifier(
            n_estimators=400,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=random_state,
            n_jobs=-1,
        ),
        "LightGBM": LGBMClassifier(
            n_estimators=400,
            learning_rate=0.05,
            num_leaves=31,
            random_state=random_state,
            n_jobs=-1,
            verbosity=-1,
        ),
    }

MODEL_SPECS = make_baseline_model_specs()
list(MODEL_SPECS.keys())


### 4.2 Grouped cross-validation helper


In [ ]:
SCORING = {
    "F1": "f1",
    "ROC_AUC": "roc_auc",
    "AP": "average_precision",
}

def run_grouped_cv(estimator, X, y, groups, n_splits=N_SPLITS_CV):
    cv = GroupKFold(n_splits=n_splits)

    cv_result = cross_validate(
        estimator=estimator,
        X=X,
        y=y,
        groups=groups,
        cv=cv,
        scoring=SCORING,
        n_jobs=1,
        return_train_score=False,
    )
    return cv_result

def summarize_cv_result(feature_name, model_name, cv_result):
    return {
        "Feature_Set": feature_name,
        "Model": model_name,
        "CV_F1_Mean": float(np.mean(cv_result["test_F1"])),
        "CV_F1_STD": float(np.std(cv_result["test_F1"])),
        "CV_ROC_AUC_Mean": float(np.mean(cv_result["test_ROC_AUC"])),
        "CV_ROC_AUC_STD": float(np.std(cv_result["test_ROC_AUC"])),
        "CV_AP_Mean": float(np.mean(cv_result["test_AP"])),
        "CV_AP_STD": float(np.std(cv_result["test_AP"])),
    }


### 4.3 Run baseline grouped CV on the development set


In [ ]:
baseline_fold_rows = []
baseline_summary_rows = []

for feature_name, feature_pack in FEATURE_SETS.items():
    print("=" * 100)
    print("Feature set:", feature_name)
    print("=" * 100)

    X_dev_feature = feature_pack["dev"]

    for model_name, estimator in MODEL_SPECS.items():
        print("Running baseline grouped CV:", model_name)

        cv_result = run_grouped_cv(
            estimator=clone(estimator),
            X=X_dev_feature,
            y=y_dev,
            groups=groups_dev,
            n_splits=N_SPLITS_CV,
        )

        baseline_summary_rows.append(
            summarize_cv_result(feature_name, model_name, cv_result)
        )

        for fold_idx in range(N_SPLITS_CV):
            baseline_fold_rows.append({
                "Feature_Set": feature_name,
                "Model": model_name,
                "Fold": fold_idx + 1,
                "F1": float(cv_result["test_F1"][fold_idx]),
                "ROC_AUC": float(cv_result["test_ROC_AUC"][fold_idx]),
                "AP": float(cv_result["test_AP"][fold_idx]),
            })

baseline_cv_folds_df = pd.DataFrame(baseline_fold_rows)
baseline_cv_summary_df = pd.DataFrame(baseline_summary_rows).sort_values(
    by=["CV_F1_Mean", "CV_ROC_AUC_Mean", "CV_AP_Mean"],
    ascending=[False, False, False]
).reset_index(drop=True)

baseline_cv_folds_df.to_csv(TABLE_DIR / "baseline_cv_folds.csv", index=False)
baseline_cv_summary_df.to_csv(TABLE_DIR / "baseline_cv_summary.csv", index=False)

baseline_cv_summary_df


### 4.4 Select the top three model-feature combinations from baseline CV


In [ ]:
top3_baseline_df = baseline_cv_summary_df.head(3).copy().reset_index(drop=True)
top3_baseline_df.to_csv(TABLE_DIR / "top3_baseline_model_feature_combinations.csv", index=False)

print("Top 3 combinations selected from development-set grouped CV:")
top3_baseline_df


## Section 5: Hyperparameter Tuning of the Top Three Models and Final Top-Two Selection

### 5.1 Tuning search spaces


In [ ]:
def make_tuning_estimator_and_grid(model_name, random_state=RANDOM_STATE):
    if model_name == "Logistic Regression":
        estimator = Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=3000, random_state=random_state)),
        ])
        param_grid = {
            "model__C": [0.1, 1.0, 5.0],
            "model__class_weight": [None, "balanced"],
        }
        return estimator, param_grid

    if model_name == "Linear SVM":
        estimator = Pipeline([
            ("scaler", StandardScaler()),
            ("model", LinearSVC(random_state=random_state)),
        ])
        param_grid = {
            "model__C": [0.1, 1.0, 5.0],
            "model__class_weight": [None, "balanced"],
        }
        return estimator, param_grid

    if model_name == "Random Forest":
        estimator = RandomForestClassifier(
            random_state=random_state,
            n_jobs=-1,
        )
        param_grid = {
            "n_estimators": [300, 500],
            "max_depth": [None, 20],
            "min_samples_split": [2, 5],
        }
        return estimator, param_grid

    if model_name == "Extra Trees":
        estimator = ExtraTreesClassifier(
            random_state=random_state,
            n_jobs=-1,
        )
        param_grid = {
            "n_estimators": [300, 500],
            "max_depth": [None, 20],
            "min_samples_split": [2, 5],
        }
        return estimator, param_grid

    if model_name == "XGBoost":
        estimator = XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=random_state,
            n_jobs=-1,
        )
        param_grid = {
            "n_estimators": [300, 500],
            "max_depth": [4, 6],
            "learning_rate": [0.03, 0.05],
            "subsample": [0.8, 1.0],
        }
        return estimator, param_grid

    if model_name == "LightGBM":
        estimator = LGBMClassifier(
            random_state=random_state,
            n_jobs=-1,
            verbosity=-1,
        )
        param_grid = {
            "n_estimators": [300, 500],
            "num_leaves": [31, 63],
            "learning_rate": [0.03, 0.05],
            "max_depth": [-1, 10],
        }
        return estimator, param_grid

    raise ValueError(f"Unknown model name: {model_name}")


### 5.2 Tune the top three combinations with grouped grid search


In [ ]:
GRID_SCORING = {
    "F1": "f1",
    "ROC_AUC": "roc_auc",
    "AP": "average_precision",
}

best_tuned_models = {}
tuned_rows = []

for _, row in top3_baseline_df.iterrows():
    feature_name = row["Feature_Set"]
    model_name = row["Model"]

    print("=" * 100)
    print("Tuning:", feature_name, " + ", model_name)
    print("=" * 100)

    X_dev_feature = FEATURE_SETS[feature_name]["dev"]
    estimator, param_grid = make_tuning_estimator_and_grid(model_name)

    grid = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        scoring=GRID_SCORING,
        refit="F1",
        cv=GroupKFold(n_splits=N_SPLITS_CV),
        n_jobs=1,
        verbose=0,
        return_train_score=False,
    )

    grid.fit(X_dev_feature, y_dev, groups=groups_dev)

    best_idx = grid.best_index_
    cv_results = grid.cv_results_

    tuned_rows.append({
        "Feature_Set": feature_name,
        "Model": model_name,
        "Best_Params": json.dumps(grid.best_params_, ensure_ascii=False),
        "Tuned_CV_F1_Mean": float(cv_results["mean_test_F1"][best_idx]),
        "Tuned_CV_F1_STD": float(cv_results["std_test_F1"][best_idx]),
        "Tuned_CV_ROC_AUC_Mean": float(cv_results["mean_test_ROC_AUC"][best_idx]),
        "Tuned_CV_ROC_AUC_STD": float(cv_results["std_test_ROC_AUC"][best_idx]),
        "Tuned_CV_AP_Mean": float(cv_results["mean_test_AP"][best_idx]),
        "Tuned_CV_AP_STD": float(cv_results["std_test_AP"][best_idx]),
    })

    best_tuned_models[(feature_name, model_name)] = grid.best_estimator_

tuned_top3_df = pd.DataFrame(tuned_rows).sort_values(
    by=["Tuned_CV_F1_Mean", "Tuned_CV_ROC_AUC_Mean", "Tuned_CV_AP_Mean"],
    ascending=[False, False, False]
).reset_index(drop=True)

tuned_top3_df.to_csv(TABLE_DIR / "tuned_top3_summary.csv", index=False)
tuned_top3_df


### 5.3 Select the final top two tuned combinations


In [ ]:
final_top2_df = tuned_top3_df.head(2).copy().reset_index(drop=True)
final_top2_df.to_csv(TABLE_DIR / "final_top2_tuned_model_feature_combinations.csv", index=False)

print("Final top 2 tuned combinations:")
final_top2_df


## Section 6: OOF Stacking with Final Holdout Test Evaluation

### 6.1 Helper functions for holdout evaluation and plots


In [ ]:
FIG_SIZE = (6, 5)
PLOT_FACE = "#ffffff"

def style_axis(ax):
    ax.set_facecolor(PLOT_FACE)
    ax.grid(True, alpha=0.4)

def get_model_scores(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1], "probability"
    if hasattr(model, "decision_function"):
        return model.decision_function(X), "decision_score"
    preds = model.predict(X)
    return preds.astype(float), "prediction"

def evaluate_holdout(y_true, y_pred, y_score, feature_name, model_name, score_type):
    return {
        "Feature_Set": feature_name,
        "Model": model_name,
        "Score_Type": score_type,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_true, y_score),
        "AP": average_precision_score(y_true, y_score),
    }

def plot_roc_curve_custom(y_true, y_score, model_name, split_name="Holdout Test", save_path=None, show=True):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    roc_auc = auc(fpr, tpr)

    fig, ax = plt.subplots(figsize=FIG_SIZE)
    style_axis(ax)
    ax.plot(fpr, tpr, linewidth=2, label=f"ROC-AUC = {roc_auc:.4f}")
    ax.plot([0, 1], [0, 1], linestyle="--", linewidth=2)
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.05, 1.05)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"{split_name} ROC Curve - {model_name}")
    ax.legend(loc="lower right")
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

    if show:
        plt.show()
    else:
        plt.close(fig)

    plt.close(fig)

def plot_confusion_figure(y_true, y_pred, model_name, split_name="Holdout Test",
                          save_path=None, display_labels=(0, 1), show=True):
    cm = confusion_matrix(y_true, y_pred)

    fig, ax = plt.subplots(figsize=FIG_SIZE)
    im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
    plt.colorbar(im, ax=ax)

    ax.set_title(f"{model_name} - {split_name} Confusion Matrix")
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_xticks(np.arange(len(display_labels)))
    ax.set_yticks(np.arange(len(display_labels)))
    ax.set_xticklabels(display_labels)
    ax.set_yticklabels(display_labels)
    ax.grid(False)

    thresh = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j, i, format(cm[i, j], "d"),
                ha="center",
                va="center",
                color="white" if cm[i, j] > thresh else "#163A70",
                fontsize=12,
            )

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

    if show:
        plt.show()
    else:
        plt.close(fig)

    plt.close(fig)

def save_holdout_plots(y_true, y_pred, y_score, feature_name, model_name):
    safe_feature = re.sub(r"[^A-Za-z0-9_]+", "_", feature_name)
    safe_model = re.sub(r"[^A-Za-z0-9_]+", "_", model_name)

    roc_path = PLOT_DIR / f"{safe_feature}__{safe_model}__roc.png"
    cm_path = PLOT_DIR / f"{safe_feature}__{safe_model}__confusion_matrix.png"

    plot_roc_curve_custom(
        y_true=y_true,
        y_score=y_score,
        model_name=model_name,
        split_name="Holdout Test",
        save_path=roc_path,
        show=True,
    )

    plot_confusion_figure(
        y_true=y_true,
        y_pred=y_pred,
        model_name=model_name,
        split_name="Holdout Test",
        save_path=cm_path,
        display_labels=(0, 1),
        show=True,
    )

    return {"roc_path": roc_path, "cm_path": cm_path}


### 6.2 Generate OOF meta-features on the development set


In [ ]:
selected_models_info = []

for _, row in final_top2_df.iterrows():
    feature_name = row["Feature_Set"]
    model_name = row["Model"]
    best_estimator = best_tuned_models[(feature_name, model_name)]

    selected_models_info.append({
        "feature_name": feature_name,
        "model_name": model_name,
        "estimator": best_estimator,
    })

print("Selected tuned models for stacking:")
for info in selected_models_info:
    print(info["feature_name"], " + ", info["model_name"])

oof_matrix = np.zeros((len(y_dev), len(selected_models_info)), dtype=np.float32)
gkf = GroupKFold(n_splits=N_SPLITS_CV)

for fold, (tr_idx, val_idx) in enumerate(gkf.split(dev_df, y_dev, groups=groups_dev), start=1):
    print(f"Generating OOF predictions - fold {fold}/{N_SPLITS_CV}")

    for model_col, info in enumerate(selected_models_info):
        X_dev_feature = FEATURE_SETS[info["feature_name"]]["dev"]

        X_tr_fold = X_dev_feature[tr_idx]
        y_tr_fold = y_dev[tr_idx]
        X_val_fold = X_dev_feature[val_idx]

        fold_model = clone(info["estimator"])
        fold_model.fit(X_tr_fold, y_tr_fold)
        val_scores, _ = get_model_scores(fold_model, X_val_fold)

        oof_matrix[val_idx, model_col] = val_scores

meta_model = LogisticRegression(max_iter=3000, random_state=RANDOM_STATE)
meta_model.fit(oof_matrix, y_dev)

oof_meta_train_df = pd.DataFrame({
    "source_id": dev_df["source_id"].to_numpy(),
    "text": dev_df["normalized_text"].to_numpy(),
    "true_label": y_dev,
})

for model_col, info in enumerate(selected_models_info):
    key = re.sub(r"[^A-Za-z0-9_]+", "_", f"{info['feature_name']}__{info['model_name']}".lower())
    oof_meta_train_df[f"oof_score_{key}"] = oof_matrix[:, model_col]

oof_meta_train_df.to_csv(TABLE_DIR / "oof_meta_train.csv", index=False)
oof_meta_train_df.head()


### 6.3 Refit the tuned top two models on the full development set


In [ ]:
refit_base_models = []
holdout_score_matrix = np.zeros((len(y_holdout), len(selected_models_info)), dtype=np.float32)
holdout_result_rows = []

for model_col, info in enumerate(selected_models_info):
    X_dev_feature = FEATURE_SETS[info["feature_name"]]["dev"]
    X_holdout_feature = FEATURE_SETS[info["feature_name"]]["holdout"]

    fitted_model = clone(info["estimator"])
    fitted_model.fit(X_dev_feature, y_dev)

    holdout_scores, score_type = get_model_scores(fitted_model, X_holdout_feature)
    holdout_preds = (holdout_scores >= 0.5).astype(int)

    result_row = evaluate_holdout(
        y_true=y_holdout,
        y_pred=holdout_preds,
        y_score=holdout_scores,
        feature_name=info["feature_name"],
        model_name=info["model_name"],
        score_type=score_type,
    )
    holdout_result_rows.append(result_row)

    prediction_df = pd.DataFrame({
        "source_id": holdout_test_df["source_id"].to_numpy(),
        "text": holdout_test_df["normalized_text"].to_numpy(),
        "true_label": y_holdout,
        "pred_label": holdout_preds,
        "score": holdout_scores,
    })

    safe_name = re.sub(r"[^A-Za-z0-9_]+", "_", f"{info['feature_name']}__{info['model_name']}")
    prediction_df.to_csv(PRED_DIR / f"{safe_name}__holdout_predictions.csv", index=False)

    saved_paths = save_holdout_plots(
        y_true=y_holdout,
        y_pred=holdout_preds,
        y_score=holdout_scores,
        feature_name=info["feature_name"],
        model_name=info["model_name"],
    )
    print("Saved ROC:", saved_paths["roc_path"])
    print("Saved Confusion Matrix:", saved_paths["cm_path"])

    holdout_score_matrix[:, model_col] = holdout_scores
    refit_base_models.append(fitted_model)

holdout_base_df = pd.DataFrame(holdout_result_rows)
holdout_base_df


### 6.4 Final stacked prediction on the holdout test set


In [ ]:
stack_feature_name = "Stacking_Selected_Top2_Models"
stack_model_name = "OOF_Stacking_Meta_LogisticRegression"

holdout_stack_scores = meta_model.predict_proba(holdout_score_matrix)[:, 1]
holdout_stack_preds = (holdout_stack_scores >= 0.5).astype(int)

stack_result_row = evaluate_holdout(
    y_true=y_holdout,
    y_pred=holdout_stack_preds,
    y_score=holdout_stack_scores,
    feature_name=stack_feature_name,
    model_name=stack_model_name,
    score_type="stacked_probability",
)

stack_result_df = pd.DataFrame([stack_result_row])

stack_prediction_df = pd.DataFrame({
    "source_id": holdout_test_df["source_id"].to_numpy(),
    "text": holdout_test_df["normalized_text"].to_numpy(),
    "true_label": y_holdout,
    "stack_score": holdout_stack_scores,
    "stack_pred": holdout_stack_preds,
})

for model_col, info in enumerate(selected_models_info):
    key = re.sub(r"[^A-Za-z0-9_]+", "_", f"{info['feature_name']}__{info['model_name']}".lower())
    stack_prediction_df[f"base_score_{key}"] = holdout_score_matrix[:, model_col]

stack_prediction_df.to_csv(PRED_DIR / "oof_stacking_holdout_predictions.csv", index=False)

saved_paths = save_holdout_plots(
    y_true=y_holdout,
    y_pred=holdout_stack_preds,
    y_score=holdout_stack_scores,
    feature_name=stack_feature_name,
    model_name=stack_model_name,
)
print("Saved ROC:", saved_paths["roc_path"])
print("Saved Confusion Matrix:", saved_paths["cm_path"])

stack_result_df


### 6.5 Final holdout comparison tables and saved outputs


In [ ]:
holdout_final_df = pd.concat([holdout_base_df, stack_result_df], ignore_index=True)
holdout_final_df = holdout_final_df.sort_values(
    by=["F1", "ROC_AUC", "AP"],
    ascending=[False, False, False],
).reset_index(drop=True)

holdout_final_df.to_csv(TABLE_DIR / "holdout_final_comparison.csv", index=False)

# Simple ranking tables from baseline CV
def prepare_sorted_cv_table(df):
    table = df.sort_values(
        by=["CV_F1_Mean", "CV_ROC_AUC_Mean", "CV_AP_Mean"],
        ascending=[False, False, False],
    ).reset_index(drop=True).copy()

    metric_cols = [
        "CV_F1_Mean", "CV_F1_STD",
        "CV_ROC_AUC_Mean", "CV_ROC_AUC_STD",
        "CV_AP_Mean", "CV_AP_STD",
    ]
    for col in metric_cols:
        table[col] = table[col].map(lambda x: round(float(x), 6))
    return table

banglabert_cv_table = prepare_sorted_cv_table(
    baseline_cv_summary_df[baseline_cv_summary_df["Feature_Set"] == "BanglaBERT_Embedding_Only"]
)
stylometry_cv_table = prepare_sorted_cv_table(
    baseline_cv_summary_df[baseline_cv_summary_df["Feature_Set"] == "Stylometric_Features_Only"]
)
fusion_cv_table = prepare_sorted_cv_table(
    baseline_cv_summary_df[baseline_cv_summary_df["Feature_Set"] == "Fusion_BanglaBERT_Plus_Stylometry"]
)
overall_cv_table = prepare_sorted_cv_table(baseline_cv_summary_df)

banglabert_cv_table.to_csv(TABLE_DIR / "baseline_ranking_banglabert_embedding_only.csv", index=False)
stylometry_cv_table.to_csv(TABLE_DIR / "baseline_ranking_stylometric_features_only.csv", index=False)
fusion_cv_table.to_csv(TABLE_DIR / "baseline_ranking_fusion_banglabert_plus_stylometry.csv", index=False)
overall_cv_table.to_csv(TABLE_DIR / "baseline_ranking_all_feature_sets_combined.csv", index=False)

summary_path = OUTPUT_ROOT / "run_summary.txt"
with open(summary_path, "w", encoding="utf-8") as f:
    f.write("Notebook completed successfully.\n\n")
    f.write(f"Dataset path: {DATA_PATH}\n")
    f.write(f"Merged rows: {len(merged_df)}\n")
    f.write(f"Development rows: {len(dev_df)}\n")
    f.write(f"Holdout test rows: {len(holdout_test_df)}\n")
    f.write(f"Embedding shape (dev): {X_dev_emb.shape}\n")
    f.write(f"Stylometry shape (dev): {X_dev_style.shape}\n")
    f.write(f"Fusion shape (dev): {X_dev_fusion.shape}\n")
    f.write(f"Stylometric feature count: {len(stylometric_feature_names)}\n\n")
    f.write("Final top 3 baseline combinations:\n")
    f.write(top3_baseline_df.to_string(index=False))
    f.write("\n\nFinal top 2 tuned combinations:\n")
    f.write(final_top2_df.to_string(index=False))
    f.write("\n\nFinal holdout comparison:\n")
    f.write(holdout_final_df.to_string(index=False))

print("=" * 100)
print("Baseline CV ranking - BanglaBERT Embedding Only")
display(banglabert_cv_table)

print("=" * 100)
print("Baseline CV ranking - Stylometric Features Only")
display(stylometry_cv_table)

print("=" * 100)
print("Baseline CV ranking - Fusion (BanglaBERT + Stylometry)")
display(fusion_cv_table)

print("=" * 100)
print("Final holdout comparison")
display(holdout_final_df)

print("Saved important files:")
print(TABLE_DIR / "baseline_cv_summary.csv")
print(TABLE_DIR / "top3_baseline_model_feature_combinations.csv")
print(TABLE_DIR / "tuned_top3_summary.csv")
print(TABLE_DIR / "final_top2_tuned_model_feature_combinations.csv")
print(TABLE_DIR / "holdout_final_comparison.csv")
print(summary_path)


In [ ]:

!cd /kaggle/working && zip -rq outputs.zip outputs
print("Zip created:", "/kaggle/working/outputs.zip")
